In [ ]:
from datetime import timedelta
from pathlib import Path

import polars as pl
import polars.selectors as cs
import plotly.express as px

from aare.pl_utils import make_quantiles
from aare_train.paths import DATA_FOLDER

# Timing comparison for run_ts and meteotest

1. Is it better to predict once at 01:00 without the input data for 01:00, which is then filled with the interpolation between 00:00 (measurement) and 02:00 (forecast),
   or is it better to just use the second predicted hour from 00:15, which had all the input data?
2. What is the alignment of hourly meteotest predictions, is it closer to FIRST or MEAN measurements?


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
import logging

logging.basicConfig(level="DEBUG")
logging.getLogger("fsspec").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("dulwich").setLevel(logging.WARNING)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
tz = "Europe/Zurich"
data_dir = DATA_FOLDER / "isolated_tasks" / "2025-05_bug-fix-eval"
actual_raw_path = data_dir / "actual_raw.parquet"
simulated_forecast_11_ing_path = data_dir / "sim_forecast_nowcasting_temp-1.1_ingestion-delay.parquet"
# live_forecast_path = data_dir / "live_forecast_nowcasting_temp-1.0.parquet"
# simulated_forecast_10_path = data_dir / "sim_forecast_nowcasting_temp-1.0.parquet"
# simulated_forecast_11_path = data_dir / "sim_forecast_nowcasting_temp-1.1.parquet"

In [ ]:
def scan_pq(path: Path):
    return pl.scan_parquet(path).with_columns(
        cs.datetime().dt.convert_time_zone(tz).dt.cast_time_unit("ns"), cs.float().cast(pl.Float32)
    )

In [ ]:
actual = scan_pq(actual_raw_path)
# MUST NOT USE SIMULATED FROM 2025-05_bug-fix-eval BECAUSE WE DIDN'T IMPLEMENT ANY LOGIC TO RESPECT THE 8min AND 15min INGESTION DELAY!
# simulated = scan_pq(simulated_forecast_11_path)
# live_forecasts = scan_pq(live_forecast_path)
# using live forecasts which (by necessity) respect that ingestion delay and run_ts 01:00 doesn't have access to the 01:00 data yet
# also doesn't work because this was the old 1.0 model which uses mean aggregation on data, so it does actually have 01:00 data, just
# by averaging the data from 00:00-00:40. Need to simulate with the ingestion delay! See notebook 25 with the temporary patch.
simulated = scan_pq(simulated_forecast_11_ing_path)

In [ ]:
evaluation_horizon = timedelta(hours=12)

In [ ]:
minute00_forecasts = simulated.filter(pl.col("run_ts").dt.minute() == 0)
minute15_forecasts = simulated.filter(pl.col("run_ts").dt.minute() == 15)

Ground rules:

- it's only about fetching forecasts between minute 00 and minute 15 (e.g. 20:00-20:15)
- data will always be fetched >= now, so if someone fetches at 20:05, the first data point must be 21:00

Options:

example: someone visits the website at 20:05

- do forecast at minute 00: because influxdb doesn't have any data for minute 00 yet, the first data point forecast will _before_ the forecast is made. lag 2 will be the next hour, e.g. run at 20:00 -> 20:00, 21:00, ...
- do forecast at minute 15: because the service wasn't run yet (remember we're between 00 and 15), the latest forecast is from the last hour. we're now past the first lag, so we need to look at the second lag. 19:15 -> 20:00, 21:00, ...

The forecast at 21:00 is either 1h old but with interpolated covariate data (lag 2 from run at 20:00), or it's 01:45 old with true measurement or forecast data (lag 2 from run at 19:15).

In [ ]:
# for minute 15 forecasts, we want to evaluate how well they perform without the first hour
# so using 00:15 for hour 02:00 instead of using 01:00.
minute15_forecasts = minute15_forecasts.filter(pl.col("time") - pl.col("run_ts") > timedelta(hours=1))

In [ ]:
# for minute 00 forecasts, we need to cut off the forecast made into the past because that would never be fetched.
# so for run_ts ~= 20:00:05, we cut off time at 20:00:00
minute00_forecasts = minute00_forecasts.filter(pl.col("time") > pl.col("run_ts"))

In [ ]:
def join_actual(df: pl.LazyFrame):
    return df.join(actual.select(time="_time", actual="temperature_bern"), on="time")


minute00_forecasts = join_actual(minute00_forecasts)
minute15_forecasts = join_actual(minute15_forecasts)

In [ ]:
def add_err(df: pl.LazyFrame):
    return df.with_columns(err=(pl.col("temp_bern") - pl.col("actual")).abs())


minute00_forecasts = add_err(minute00_forecasts)
minute15_forecasts = add_err(minute15_forecasts)

In [ ]:
def filter_horizon(df: pl.LazyFrame, horizon: timedelta):
    return df.filter(pl.col("time") - pl.col("run_ts") < horizon)


minute00_forecasts = filter_horizon(minute00_forecasts, evaluation_horizon)
# must shift horizon by 1 because we want to simulate treating an old forecast like a new one basically
minute15_forecasts = filter_horizon(minute15_forecasts, evaluation_horizon + timedelta(hours=1))

In [ ]:
def add_first_hour(df: pl.LazyFrame):
    # for minute 00 forecasts (with removed past/same-hour forecast), the first hour should be the next hour so 20:00 -> 21:00
    # for minute 15 forecasts (with skipped lag 1), the first hour should be 1h45min later, so 19:15 -> 21:00
    return df.with_columns(first_hour=pl.col("time").min().over("run_ts"))


minute00_forecasts = add_first_hour(minute00_forecasts)
minute15_forecasts = add_first_hour(minute15_forecasts)

In [ ]:
# confirm both dfs have the same horizons per run, in line with the specified evaluation_horizon
minute15_forecasts.group_by("run_ts").agg(pl.count("time").alias("horizon")).select(
    pl.col("horizon").value_counts()
).unnest("horizon").collect()

In [ ]:
minute00_forecasts.group_by("run_ts").agg(pl.count("time").alias("horizon")).select(
    pl.col("horizon").value_counts()
).unnest("horizon").collect()

In [ ]:
# confirm that horizon 1 is cut off, so if run_ts=19:15, first_hour should be 21:00
minute15_forecasts.collect()

In [ ]:
# confirm that forecasts at 20:00 have 21:00 as first hour (because we don't care about forecast made into the past)
minute00_forecasts.collect()

In [ ]:
def agg_err(df: pl.LazyFrame):
    return df.group_by("first_hour").agg(make_quantiles("err", quantiles=[0.25, 0.75, 0.95]))


agg_errors = (
    agg_err(minute00_forecasts)
    .join(agg_err(minute15_forecasts), on="first_hour", suffix="_m15")
    .sort("first_hour")
    .collect()
)
agg_errors

In [ ]:
m15_suffix = "_m15"
suffixes = ["", "_q25.0", "_q75.0", "_q95.0"]
cols = ["err_diff" + s for s in suffixes]
df = agg_errors.select(
    "first_hour", *[(pl.col("err" + s + m15_suffix) - pl.col("err" + s)).alias("err_diff" + s) for s in suffixes]
)
px.line(df, x="first_hour", y=cols)
# line chart shows, neither is strictly better. fluctuations are high, there's no real trend visible.

In the following table we can see which performs better when looking only at the horizon specified above.

Negative err_diff means that the m15 forecast error was **lower** than the 00 forecast -> using the old forecast from last hour performed better than using the new one.

Positive: err_m15 > err_m00

In [ ]:
df.describe()

In [ ]:
px.violin(df, x=cols, box=True, range_x=(-0.4, 0.4))

**Conclusion**: Making a forecast at 00:00 with interpolated air temp is actually better on average than using the 1h45m old one.
This went against everything I believed, but I triple checked the calculations, made sure to simulate forecasts correctly with ingestion delay,
checked many horizons like 1, 3, 6 and 12, etc. Note, there are plenty of times when the m15 forecast is better than the m00 one, but it's
less than 50% of the time and the distribution shows that even for the 95% error quantile (basically the worst errors of a run within the specified horizon)
there are more runs where m15 has a higher error than m00.

In the process I did the same analysis with live forecasts and simulated forecasts without ingestion delay
(both should not really be looked at too closely because they are likely unrepresentative), but they also showed that the m15 error is
on average higher than the m00 one.

## Decision on forecast schedule

Combining this knowledge together with the meteotest update schedule from notebook 23, I will configure the forecast schedule as follows:

Example: run_ts >=20:00,<21:00, first relevant forecast point is 21:00

- Minute 01 (20:01):
  - First forecast point: 20:00
  - Latest past hydro/temperature value: 19:00
  - Latest past smn/tt value: 19:00
  - Notes:
    - Will forecast into the past because of missing target
    - Will use interpolated air temp feature because of missing 20:00 value
    - Was afraid that interpolation would make for worse forecasts, but as seen above that's not the case.
  - Why: Can make use of meteotest updates every 6 and 12 hours
- Minute 09 (20:09):
  - First forecast point: 21:00
  - Latest past hydro/temperature value: 20:00
  - Latest past smn/tt value: 19:00
  - Notes:
    - Will use interpolated air temp feature because of missing 20:00 value
    - Didn't evaluate specifically, but from watching live forecasts, this seems much better than the 00 forecast because of the updated target, so should be worth.
  - Why: Can make use of updated last hydro/temperature in influx db which arrives at minute ~8, so no more forecast into the past
- Minute 16 (20:16):
  - First forecast point: 21:00
  - Latest past hydro/temperature value: 20:00
  - Latest past smn/tt value: 20:00
  - Notes:
    - Latest available data
  - Why: Can make use of updated last smn/tt in influx db which arrives at minute ~15, so no more interpolation for air temp
- Minute 46 (20:46):
  - First forecast point: 21:00
  - Latest past hydro/temperature value: 20:00
  - Latest past smn/tt value: 20:00
  - Why: Can make use of meteotest updates every hour between minute 30 and minute 45 (not sure about exact timing).

This is definitely the maximum that makes sense and seizes every moment new data becomes available.
In the 30min gap every forecast would be exactly the same as far as we know.